Pipeline steps:
  1. Load raw 1 Hz data.
  2. Run elbow study on training data to choose number of regimes (k).
  3. Fit RegimeClusterer (k-means++) on training data only.
  4. Aggregate both splits to one row per (unit, cycle), with
     regime-conditioned sensor features.
  5. Apply piecewise-linear RUL transformation.
  6. Cache results as Parquet.

In [1]:
import sys
sys.path.append("..")

from src.logging_config import setup_logging
setup_logging()

import logging
import matplotlib.pyplot as plt
from pathlib import Path

from src.data import load_split, DATA_PATH
from src.features import (
    RegimeClusterer, elbow_study, aggregate_to_cycle_level,
    piecewise_linear_rul, save_parquet, W_VARS,
)

logger = logging.getLogger("phase3")
FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

Load Raw Data

In [2]:
H5 = next(DATA_PATH.glob("*.h5"))
dev = load_split(H5, split="dev")
test = load_split(H5, split="test")

#Cherry pick sensor columns
xs_cols = [c for c in dev.columns
           if c not in W_VARS + ["unit", "cycle", "Fc", "hs", "RUL"]]
logger.info("Sensor columns (%d): %s", len(xs_cols), xs_cols)

22:16:37 | INFO    | src.data | Loading split=dev from N-CMAPSS_DS02-006.h5
22:16:38 | INFO    | src.data | Loaded split=dev: shape=(5263447, 23), memory=0.48 GB
22:16:38 | INFO    | src.data | Loading split=test from N-CMAPSS_DS02-006.h5
22:16:38 | INFO    | src.data | Loaded split=test: shape=(1253743, 23), memory=0.12 GB
22:16:38 | INFO    | phase3 | Sensor columns (14): ['T24', 'T30', 'T48', 'T50', 'P15', 'P2', 'P21', 'P24', 'Ps30', 'P40', 'P50', 'Nf', 'Nc', 'Wf']


Elbow study

In [3]:
elbow = elbow_study(dev, k_values=tuple(range(2, 11)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(elbow["k"], elbow["inertia"], marker="o", linewidth=1.8)
ax.set_xlabel("Number of regimes (k)")
ax.set_ylabel("Within-cluster sum of squares (inertia)")
ax.set_title("Elbow method for regime count")
ax.grid(alpha=0.3)
fig.tight_layout()
elbow_path = FIG_DIR / "elbow_regimes.png"
fig.savefig(elbow_path, dpi=150)
plt.close(fig)
logger.info("Saved %s", elbow_path)

22:16:38 | INFO    | src.features | Elbow study: k=(2, 3, 4, 5, 6, 7, 8, 9, 10), sample_size=200000
22:16:38 | INFO    | src.features |   k=2 -> inertia=324209
22:16:38 | INFO    | src.features |   k=3 -> inertia=240912
22:16:39 | INFO    | src.features |   k=4 -> inertia=181823
22:16:39 | INFO    | src.features |   k=5 -> inertia=144247
22:16:40 | INFO    | src.features |   k=6 -> inertia=129134
22:16:40 | INFO    | src.features |   k=7 -> inertia=115432
22:16:41 | INFO    | src.features |   k=8 -> inertia=103320
22:16:42 | INFO    | src.features |   k=9 -> inertia=94938
22:16:43 | INFO    | src.features |   k=10 -> inertia=86714
22:16:43 | INFO    | phase3 | Saved ../reports/figures/elbow_regimes.png


Fit regime clusterer

In [4]:
N_REGIMES = 5
clusterer = RegimeClusterer(n_regimes=N_REGIMES, random_state=42).fit(dev)

#Sanity check
dev_sample = dev.sample(50_000, random_state=0)
dev_labels_sample = clusterer.predict(dev_sample)
logger.info("Regime characterization (means of W vars per regime):\n%s",
            clusterer.describe_regimes(dev_sample, dev_labels_sample))

22:16:43 | INFO    | src.features | Fitting RegimeClusterer with k=5 on up to 500000 samples
22:16:45 | INFO    | src.features | Cluster inertia: 359888
22:16:45 | INFO    | phase3 | Regime characterization (means of W vars per regime):
                 alt             Mach               TRA                 T2  \
                mean   size      mean   size       mean   size        mean   
regime                                                                       
0       21346.648438   5808  0.613343   5808  49.671696   5808  475.991272   
1       12582.128906   7431  0.508421   7431  45.730442   7431  498.488586   
2       28604.011719  16623  0.699231  16623  77.021011  16623  457.468719   
3       22959.320312  12330  0.643236  12330  75.491371  12330  472.982269   
4       15045.383789   7808  0.547629   7808  74.662376   7808  492.971771   

               
         size  
regime         
0        5808  
1        7431  
2       16623  
3       12330  
4        7808  


Predict regimes for full splits


In [5]:
logger.info("Predicting regimes for dev (%d rows)...", len(dev))
dev_labels = clusterer.predict(dev)
logger.info("Predicting regimes for test (%d rows)...", len(test))
test_labels = clusterer.predict(test)

22:16:45 | INFO    | phase3 | Predicting regimes for dev (5263447 rows)...
22:16:45 | INFO    | phase3 | Predicting regimes for test (1253743 rows)...


Cycle level aggregation

In [6]:
dev_cycle = aggregate_to_cycle_level(
    dev, sensor_cols=xs_cols,
    regime_labels=dev_labels, n_regimes=clusterer.n_regimes,
)
test_cycle = aggregate_to_cycle_level(
    test, sensor_cols=xs_cols,
    regime_labels=test_labels, n_regimes=clusterer.n_regimes,
)

22:16:45 | INFO    | src.features | Aggregating 5263447 rows to cycle level...
22:16:48 | INFO    | src.features | Cycle-level features: 446 rows × 150 columns
22:16:48 | INFO    | src.features | Aggregating 1253743 rows to cycle level...
22:16:49 | INFO    | src.features | Cycle-level features: 202 rows × 150 columns


Piecewise-linear RUL transformation

In [7]:
dev_cycle["RUL_pw"] = piecewise_linear_rul(dev_cycle["RUL"], r_max=50)
test_cycle["RUL_pw"] = piecewise_linear_rul(test_cycle["RUL"], r_max=50)

logger.info("Final feature counts: dev=%d cols, test=%d cols", dev_cycle.shape[1], test_cycle.shape[1])
logger.info("Sample feature names: %s", list(dev_cycle.columns[:20]))

22:16:49 | INFO    | phase3 | Final feature counts: dev=151 cols, test=151 cols
22:16:49 | INFO    | phase3 | Sample feature names: ['T24_mean', 'T24_std', 'T24_min', 'T24_max', 'T30_mean', 'T30_std', 'T30_min', 'T30_max', 'T48_mean', 'T48_std', 'T48_min', 'T48_max', 'T50_mean', 'T50_std', 'T50_min', 'T50_max', 'P15_mean', 'P15_std', 'P15_min', 'P15_max']


Cache

In [8]:
save_parquet(dev_cycle.reset_index(),  "dev_cycle_features")
save_parquet(test_cycle.reset_index(), "test_cycle_features")

22:16:50 | INFO    | src.features | Wrote /Users/alan/Desktop/ML_Final_Project/data/interim/dev_cycle_features.parquet (0.4 MB)
22:16:50 | INFO    | src.features | Wrote /Users/alan/Desktop/ML_Final_Project/data/interim/test_cycle_features.parquet (0.2 MB)


PosixPath('/Users/alan/Desktop/ML_Final_Project/data/interim/test_cycle_features.parquet')